In [40]:

import sys
print(f'Python: {sys.version}')

Python: 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]


In [4]:
import pandas as pd
import numpy as np
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import RegressionModel
from darts.metrics import mae, rmse
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import plotly.graph_objects as go

import xgboost as xgb
import lightgbm as lgb
print(f'pandas:   {pd.__version__}')
print(f'xgboost:  {xgb.__version__}')
print(f'lightgbm: {lgb.__version__}')
print('All imports OK')

pandas:   3.0.1
xgboost:  2.1.3
lightgbm: 4.6.0
All imports OK


In [5]:
RANDOM_STATE = 42
INPUT_WINDOW = 12   # months of history used as input
HORIZON      = 6    # months ahead to forecast
target_col   = 'resistance_pct'

# Update this path to match your system
DATA_PATH   = '/Users/anisa/OneDrive/Desktop/fyp/notebooks/combined_features.csv'
OUTPUT_PATH = '/Users/anisa/OneDrive/Desktop/fyp/data/'

Load Data

In [6]:
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['series_id', 'date'])

print(f'Total rows:    {len(df)}')
print(f'Series count:  {df["series_id"].nunique()}')
print(f'Date range:    {df["date"].min().date()} → {df["date"].max().date()}')
print(f'NaN in target: {df[target_col].isna().sum()}')
df.head()

Total rows:    9561
Series count:  207
Date range:    2021-08-01 → 2025-06-01
NaN in target: 0


,geography,stratum,date,organism,resistance_pct,series_id,model_region,prescribing_rate,month,sin_month,...,lag_12,presc_lag_1,presc_lag_3,presc_lag_6,trend,prescribing_x_month,roll_mean_3,roll_mean_6,roll_std_6,presc_roll_mean_3
0,East Midlands,Amikacin,2021-10-01,E-coli,1.52,E-coli__Amikacin__East Midlands,Midlands,1.34,10,-8.660254e-01,...,0.53,1.31,1.28,1.25,0.21,13.40,1.853333,1.788333,0.191668,1.296667
1,East Midlands,Amikacin,2021-11-01,E-coli,1.17,E-coli__Amikacin__East Midlands,Midlands,1.37,11,-5.000000e-01,...,0.92,1.34,1.30,1.25,-0.55,15.07,1.813333,1.776667,0.207621,1.316667
2,East Midlands,Amikacin,2021-12-01,E-coli,0.72,E-coli__Amikacin__East Midlands,Midlands,1.41,12,-2.449294e-16,...,0.93,1.37,1.31,1.26,-0.68,16.92,1.513333,1.696667,0.325310,1.340000
3,East Midlands,Amikacin,2022-01-01,E-coli,1.90,E-coli__Amikacin__East Midlands,Midlands,1.41,1,5.000000e-01,...,0.53,1.41,1.34,1.28,-0.80,1.41,1.136667,1.495000,0.486734,1.373333
4,East Midlands,Amikacin,2022-02-01,E-coli,2.43,E-coli__Amikacin__East Midlands,Midlands,1.43,2,8.660254e-01,...,0.17,1.41,1.37,1.30,0.73,2.86,1.263333,1.538333,0.513085,1.396667


 Build TimeSeries & Covariates

In [7]:
series_raw = {}
cov_raw    = {}

for sid, g in df.groupby('series_id'):
    g = g.set_index('date')

    target_ts = TimeSeries.from_series(g[target_col], freq='MS')

    numeric_cols = g.select_dtypes(include=['number']).columns.tolist()
    feat_cols    = [c for c in numeric_cols if c != target_col]

    cov_ts = TimeSeries.from_dataframe(g[feat_cols], freq='MS') if feat_cols else None

    series_raw[sid] = target_ts
    cov_raw[sid]    = cov_ts

# Series length check — mirrors DL notebook
min_len = min(len(ts) for ts in series_raw.values())
print(f'Shortest series: {min_len} months')
print(f'Minimum needed (window={INPUT_WINDOW} + horizon={HORIZON}): {INPUT_WINDOW + HORIZON} months')
if min_len < INPUT_WINDOW + HORIZON:
    print('WARNING: some series are too short — they will be skipped during backtest')
else:
    print('All series long enough — OK')

Shortest series: 45 months
Minimum needed (window=12 + horizon=6): 18 months
All series long enough — OK


In [8]:
from darts.dataprocessing.transformers import Scaler

target_scalers = {}
series_scaled  = {}

for sid, ts in series_raw.items():
    scaler = Scaler()
    series_scaled[sid] = scaler.fit_transform(ts)
    target_scalers[sid] = scaler

Model Builders


- **Direct** (`output_chunk_length=HORIZON`): predicts all 6 months in one shot — no error accumulation
- **Recursive** (`output_chunk_length=1`): predicts 1 month, feeds that back as input, repeats 6×


In [9]:
def build_xgb_direct():
    return RegressionModel(
        lags=INPUT_WINDOW,
        lags_past_covariates=INPUT_WINDOW,
        output_chunk_length=HORIZON,          # direct: all 6 months at once
        model=XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

def build_xgb_recursive():
    return RegressionModel(
        lags=INPUT_WINDOW,
        lags_past_covariates=INPUT_WINDOW,
        output_chunk_length=1,                # recursive: 1 step at a time
        model=XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

def build_lgbm_direct():
    return RegressionModel(
        lags=INPUT_WINDOW,
        lags_past_covariates=INPUT_WINDOW,    # FIX: was missing in original
        output_chunk_length=HORIZON,
        model=LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            verbose=-1
        )
    )

def build_lgbm_recursive():
    return RegressionModel(
        lags=INPUT_WINDOW,
        lags_past_covariates=INPUT_WINDOW,    # FIX: was missing in original
        output_chunk_length=1,
        model=LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            verbose=-1
        )
    )

print('Model builders defined: XGB-Direct, XGB-Recursive, LGBM-Direct, LGBM-Recursive')

Model builders defined: XGB-Direct, XGB-Recursive, LGBM-Direct, LGBM-Recursive


 Backtest Function


In [10]:
from darts import concatenate

start_bt = pd.Timestamp('2022-07-01')
approx_folds = (pd.Timestamp('2025-06-01') - start_bt).days // 30 // HORIZON
print(f'Backtest start:         {start_bt.date()}')
print(f'Approx folds per series: {approx_folds}')


def backtest_ml(build_model_fn, series, covariates, start_date=start_bt):
    """
    Rolling backtest using Darts historical_forecasts.
    forecast_horizon=HORIZON, stride=HORIZON → non-overlapping 6-month folds.
    """
    metrics   = {}
    forecasts = {}

    for sid, ts in series.items():
        cov = covariates[sid]

        # Skip series too short to backtest
        if len(ts) < INPUT_WINDOW + HORIZON:
            print(f'  Skipping {sid} — too short ({len(ts)} months)')
            continue

        # Use fixed start or fall back to 70% split
        if start_date in ts.time_index:
            actual_start = start_date
        else:
            start_idx    = int(len(ts) * 0.7)
            actual_start = ts.time_index[start_idx]

        try:
            model = build_model_fn()
            model.fit(series=ts, past_covariates=cov)

            bt = model.historical_forecasts(
                series=ts,
                past_covariates=cov,
                start=actual_start,
                forecast_horizon=HORIZON,   # FIX: was 1
                stride=HORIZON,             # FIX: was 1 — non-overlapping windows
                retrain=True,
                verbose=False,
                last_points_only=False      # keep full 6-month forecast per fold
            )

            # Flatten list of per-fold forecasts into single TimeSeries
            if isinstance(bt, list):
                bt = concatenate(bt)

            bt_intersect = bt.slice_intersect(ts)
            m_mae  = float(mae(ts,  bt_intersect))
            m_rmse = float(rmse(ts, bt_intersect))

            # NaN guard — mirrors DL notebook clean_metrics()
            if pd.isna(m_mae) or pd.isna(m_rmse):
                print(f'  NaN metrics for {sid} — skipping')
                continue

            metrics[sid]   = {'mae': m_mae, 'rmse': m_rmse}
            forecasts[sid] = bt_intersect

        except Exception as e:
            print(f'  ERROR on {sid}: {e}')
            continue

    print(f'  Done — {len(metrics)} series evaluated')
    return metrics, forecasts


print('backtest_ml() defined.')

Backtest start:         2022-07-01
Approx folds per series: 5
backtest_ml() defined.


 Run All Backtests

In [11]:
xgb_direct_metrics, xgb_direct_forecasts = backtest_ml(build_xgb_direct,  series_scaled, cov_raw)
xgb_rec_metrics,    xgb_rec_forecasts    = backtest_ml(build_xgb_recursive, series_scaled, cov_raw)
lgbm_direct_metrics, lgbm_direct_forecasts = backtest_ml(build_lgbm_direct,  series_scaled, cov_raw)
lgbm_rec_metrics,   lgbm_rec_forecasts   = backtest_ml(build_lgbm_recursive, series_scaled, cov_raw)

`start` time `2022-07-01 00:00:00` is before the first predictable/trainable historical forecasting point for series at index: 0. Using the first historical forecasting point `2023-07-01 00:00:00` that lies a round-multiple of `stride=6` ahead of `start`. To hide these warnings, set `show_warnings=False`.
`start` time `2022-07-01 00:00:00` is before the first predictable/trainable historical forecasting point for series at index: 0. Using the first historical forecasting point `2023-07-01 00:00:00` that lies a round-multiple of `stride=6` ahead of `start`. To hide these warnings, set `show_warnings=False`.
`start` time `2022-07-01 00:00:00` is before the first predictable/trainable historical forecasting point for series at index: 0. Using the first historical forecasting point `2023-07-01 00:00:00` that lies a round-multiple of `stride=6` ahead of `start`. To hide these warnings, set `show_warnings=False`.
`start` time `2022-07-01 00:00:00` is before the first predictable/trainable hi

  Done — 207 series evaluated


`start` time `2022-07-01 00:00:00` is before the first predictable/trainable historical forecasting point for series at index: 0. Using the first historical forecasting point `2023-01-01 00:00:00` that lies a round-multiple of `stride=6` ahead of `start`. To hide these warnings, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`start` time `2022-07-01 00:00:00` is before the first predictable/trainable historical forecasting point for series at index: 0. Using the first historical forecasting point `2023-01-01 00:00:00` that lies a round-multiple of `stride=6` ahead of `start`. To hide these warnings, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-

  Done — 207 series evaluated


`start` time `2022-07-01 00:00:00` is before the first predictable/trainable historical forecasting point for series at index: 0. Using the first historical forecasting point `2023-07-01 00:00:00` that lies a round-multiple of `stride=6` ahead of `start`. To hide these warnings, set `show_warnings=False`.
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\vali

  Done — 207 series evaluated


c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with f

  Done — 207 series evaluated


c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with f

Compare Results

In [12]:
def metrics_to_df(metrics_dict, model_name):
    df_m = pd.DataFrame.from_dict(metrics_dict, orient='index')
    df_m['model'] = model_name
    return df_m

results = pd.concat([
    metrics_to_df(xgb_direct_metrics,  'XGB-Direct'),
    metrics_to_df(xgb_rec_metrics,     'XGB-Recursive'),
    metrics_to_df(lgbm_direct_metrics, 'LGBM-Direct'),
    metrics_to_df(lgbm_rec_metrics,    'LGBM-Recursive'),
])

summary = results.groupby('model')[['mae', 'rmse']].mean().sort_values('mae')
print('=== MEAN METRICS ACROSS ALL SERIES (6-month horizon, stride=6) ===')
print(summary)

# Direct vs Recursive per family
for family in ['XGB', 'LGBM']:
    direct_mae = summary.loc[f'{family}-Direct', 'mae']
    rec_mae    = summary.loc[f'{family}-Recursive', 'mae']
    improvement = (rec_mae - direct_mae) / rec_mae * 100
    print(f'\n{family}: Direct is {improvement:.1f}% better MAE than Recursive')

summary['strategy'] = summary.index.map(lambda x: 'Direct' if 'Direct' in x else 'Recursive')
print('\n=== DIRECT vs RECURSIVE (averaged across families) ===')
print(summary.groupby('strategy')[['mae', 'rmse']].mean())

=== MEAN METRICS ACROSS ALL SERIES (6-month horizon, stride=6) ===
                     mae      rmse
model                             
LGBM-Recursive  0.216642  0.266697
LGBM-Direct     0.221553  0.270321
XGB-Recursive   0.245633  0.305680
XGB-Direct      0.263920  0.323440

XGB: Direct is -7.4% better MAE than Recursive

LGBM: Direct is -2.3% better MAE than Recursive

=== DIRECT vs RECURSIVE (averaged across families) ===
                mae      rmse
strategy                     
Direct     0.242736  0.296880
Recursive  0.231138  0.286189


 Train Final Models on Full Data

In [13]:
def train_final_models(build_model_fn, series, covariates):
    """Train on full series — no train/test split — for future forecasting."""
    final_models = {}
    for sid, ts in series.items():
        cov = covariates[sid]
        model = build_model_fn()
        model.fit(series=ts, past_covariates=cov)
        final_models[sid] = model
    return final_models

xgb_final_models  = train_final_models(build_xgb_direct,  series_scaled, cov_raw)
lgbm_final_models = train_final_models(build_lgbm_direct, series_scaled, cov_raw)
print('Final models trained.')

Final models trained.


In [30]:
def forecast_to_df(final_models, series, covariates, scalers, model_label):
    rows = []

    for sid, model in final_models.items():
        ts     = series[sid]    # scaled
        cov_ts = covariates[sid]

        # Inverse transform actuals back to real %
        ts_real = scalers[sid].inverse_transform(ts)
        ts_df   = ts_real.to_dataframe().reset_index()
        ts_df.columns        = ['date', 'value']
        ts_df['is_forecast'] = False
        ts_df['series_id']   = sid

        if sid in scenarios:
            future_cov = scenarios[sid]['baseline']
            full_cov   = cov_ts.append(future_cov) if cov_ts is not None else None
        else:
            full_cov = cov_ts

        fc_scaled = model.predict(n=HORIZON, past_covariates=full_cov)

        # Inverse transform forecast back to real %
        fc_real = scalers[sid].inverse_transform(fc_scaled)
        fc_df   = fc_real.to_dataframe().reset_index()
        fc_df.columns        = ['date', 'value']
        fc_df['is_forecast'] = True
        fc_df['series_id']   = sid

        rows.append(pd.concat([ts_df, fc_df], ignore_index=True))

    combined = pd.concat(rows, ignore_index=True)
    combined['model'] = model_label
    combined['date']  = pd.to_datetime(combined['date'])
    return combined.sort_values(['series_id', 'date'])


def build_forecast_only_table(scenario_results, sid):
    """
    Wide-format scenario table: date | baseline | minus20 | plus20 | series_id
    """
    rows = []
    for scen_name, ts in scenario_results[sid].items():
        d = ts.to_dataframe().reset_index()
        d.columns = ['date', scen_name]
        d['series_id'] = sid
        rows.append(d)

    df_merged = rows[0]
    for d in rows[1:]:
        df_merged = df_merged.merge(d, on=['date', 'series_id'], how='left')
    return df_merged


print('Helper functions defined.')

Helper functions defined.


In [31]:
df_all_xgb  = forecast_to_df(xgb_final_models,  series_scaled, cov_raw, target_scalers, model_label='xgb')
df_all_lgbm = forecast_to_df(lgbm_final_models, series_scaled, cov_raw, target_scalers, model_label='lgbm')

print(f'XGB  df — shape: {df_all_xgb.shape},  NaN: {df_all_xgb.isna().sum().sum()}')
print(f'LGBM df — shape: {df_all_lgbm.shape}, NaN: {df_all_lgbm.isna().sum().sum()}')
df_all_xgb.tail()

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\si

XGB  df — shape: (10803, 5),  NaN: 0
LGBM df — shape: (10803, 5), NaN: 0


c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

c:\Users\anisa\miniconda3\envs\fyp_env\Lib\si

,date,value,is_forecast,series_id,model
10798,2025-08-01,16.313574,True,e-faecium__Glycopeptides__Yorkshire and Humber,xgb
10799,2025-09-01,15.849689,True,e-faecium__Glycopeptides__Yorkshire and Humber,xgb
10800,2025-10-01,22.173853,True,e-faecium__Glycopeptides__Yorkshire and Humber,xgb
10801,2025-11-01,22.214975,True,e-faecium__Glycopeptides__Yorkshire and Humber,xgb
10802,2025-12-01,17.652571,True,e-faecium__Glycopeptides__Yorkshire and Humber,xgb


In [ ]:
def plot_continuous(df_all, sid, model_name='Model'):
    df_sid   = df_all[df_all['series_id'] == sid].sort_values('date')
    actual   = df_sid[df_sid['is_forecast'] == False]
    forecast = df_sid[df_sid['is_forecast'] == True]

    # Anchor forecast to last actual point for visual continuity
    if not actual.empty and not forecast.empty:
        forecast = pd.concat([actual.tail(1), forecast], ignore_index=True)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=actual['date'], y=actual['value'],
        mode='lines', name='Actual', line=dict(color='blue')
    ))
    fig.add_trace(go.Scatter(
        x=forecast['date'], y=forecast['value'],
        mode='lines', name=f'{model_name} Forecast',
        line=dict(color='red', dash='dash')
    ))
    fig.update_layout(
        title=f'{sid} — {model_name} Continuous Forecast',
        xaxis=dict(title='Date', type='date', tickformat='%b %Y', dtick='M3'),
        yaxis=dict(title='Resistance (%)'),
        template='plotly_white', showlegend=True
    )
    fig.show()



sid = 'e-faecium__Glycopeptides__East of England'
plot_continuous(df_all_xgb,  sid, model_name='XGBoost')
plot_continuous(df_all_lgbm, sid, model_name='LightGBM')

## Cell 17 — Export All Outputs

In [39]:
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- Actuals ---
df_actuals = df[['date', 'series_id', target_col]].copy()
df_actuals.rename(columns={target_col: 'value'}, inplace=True)
df_actuals.to_csv(f'{OUTPUT_PATH}ml_actuals.csv', index=False)
print('Saved: ml_actuals.csv')

# --- Continuous forecasts (actuals + forecast rows) ---
df_all_xgb.to_csv(f'{OUTPUT_PATH}xgb_all.csv', index=False)
df_all_lgbm.to_csv(f'{OUTPUT_PATH}lgbm_all.csv', index=False)
print('Saved: xgb_all.csv, lgbm_all.csv')

# --- Combined future-only forecasts ---
all_ml_forecasts = pd.concat([
    df_all_xgb[df_all_xgb['is_forecast']   == True],
    df_all_lgbm[df_all_lgbm['is_forecast'] == True],
], ignore_index=True)
all_ml_forecasts.to_csv(f'{OUTPUT_PATH}all_ml_forecasts.csv', index=False)
print('Saved: all_ml_forecasts.csv')

# --- Metrics ---
summary.to_csv(f'{OUTPUT_PATH}ml_model_comparison_metrics.csv')
results.to_csv(f'{OUTPUT_PATH}ml_per_series_metrics.csv')
print('Saved: ml_model_comparison_metrics.csv, ml_per_series_metrics.csv')
print(f'\nAll files saved to: {OUTPUT_PATH}')

Saved: ml_actuals.csv
Saved: xgb_all.csv, lgbm_all.csv
Saved: all_ml_forecasts.csv
Saved: ml_model_comparison_metrics.csv, ml_per_series_metrics.csv

All files saved to: /Users/anisa/OneDrive/Desktop/fyp/data/
